# Thesis Colab
---

## 1a. Download the required deps...

In [ ]:
# Dependencies - Run this in a code cell
!pip install -q -U torch transformers bitsandbytes accelerate fastapi uvicorn python-multipart nest-asyncio
!pip install huggingface_hub uv

## 1b. Verify System

In [ ]:
# Quick Check: Verify the GPU
import torch

# 1. Check Availability and Device Name
if torch.cuda.is_available():
    print(f"CUDA Available: {torch.cuda.is_available()}")
    device_name = torch.cuda.get_device_name(0)
    print(f"Current Device: {device_name}")

    # 2. Get VRAM Details via PyTorch
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9  # Convert bytes to GB
    print(f"Total VRAM: {total_vram:.2f} GB")

    # 3. Double check with system command (gives a visual table)
    print("\n--- NVIDIA-SMI Output ---")
    !nvidia-smi
else:
    print("❌ CUDA is NOT available. Go to Runtime > Change runtime type > Select T4 GPU.")

## 1c. Download and Load Model (4-bit Quantized)

In [ ]:
# Setup HuggingFace
from google.colab import userdata
from huggingface_hub import login

# If you stored your token in Colab Secrets (key: 'HF_TOKEN')
login(token=userdata.get('HF_TOKEN'))

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

# Model ID
model_id = "meta-llama/Llama-3.1-8B-Instruct"

# Quantization Config (4-bit) to save memory
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token # Fix for Llama 3

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

## 2. Setup Cloudflared

In [ ]:
# Setup Clouflared
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

## 3. Setup Backend and FastAPI

In [ ]:
%%writefile app.py
from fastapi import FastAPI, HTTPException, status
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import os

app = FastAPI(title="Adaptive Quiz Generator API")

# IMPORTANT: Enable CORS so your React app can talk to this server
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # For production, replace with your specific React URL
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Define the request body to match your React 'fetch' call
class LoginRequest(BaseModel):
    username: str
    password: str
    secret: str
    salt: str

# Mock database of users and passwords for testing
# In your thesis project, these would be in MongoDB
USER_DB = {
    "admin123": "admin",
    "regular123": "regular",
    "trial123": "free_trial"
}

# The specific secret you personally created
MY_PERSONAL_SECRET = "l49dVcs3GoKDZCRw60KBqpgOlkXsihoct54C35k02VE="

# --- GLOBAL MODEL LOADING ---
# We load it here again or import it if running as a script.
# For Colab specifically, it's often easier to rely on the global variables
# if running strictly inside the notebook, but for a true backend,
# we should load it within the app context or use a lifespan handler.
# For simplicity in this demo, we assume the model is loaded in the global scope
# of the notebook, but to run "uvicorn app:app", we need to load it inside.

# model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

# print("Loading Model in API...")
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16
# )

# tokenizer = AutoTokenizer.from_pretrained(model_id)
# tokenizer.pad_token = tokenizer.eos_token
# model = AutoModelForCausalLM.from_pretrained(
#     model_id,
#     quantization_config=bnb_config,
#     device_map="auto"
# )

class GenerateRequest(BaseModel):
    prompt: str
    max_tokens: int = 512
    temperature: float = 0.7


# @app.post("/generate")
# def generate_text(req: GenerateRequest):
#     try:
#         messages = [
#             {"role": "system", "content": "You are a helpful assistant for generating quiz questions."},
#             {"role": "user", "content": req.prompt},
#         ]

#         input_ids = tokenizer.apply_chat_template(
#             messages,
#             add_generation_prompt=True,
#             return_tensors="pt"
#         ).to(model.device)

#         terminators = [
#             tokenizer.eos_token_id,
#             tokenizer.convert_tokens_to_ids("<|eot_id|>")
#         ]

#         outputs = model.generate(
#             input_ids,
#             max_new_tokens=req.max_tokens,
#             eos_token_id=terminators,
#             do_sample=True,
#             temperature=req.temperature,
#             top_p=0.9,
#         )

#         response = outputs[0][input_ids.shape[-1]:]
#         decoded_text = tokenizer.decode(response, skip_special_tokens=True)

#         return {"response": decoded_text}

#     except Exception as e:
#         raise HTTPException(status_code=500, detail=str(e))

Writing app.py


## 4. Run Backend Server

In [ ]:
import nest_asyncio
import uvicorn
import threading

# 1. Allow nested event loops (required for Colab)
nest_asyncio.apply()

# 2. Define a function to run the server
def run_server():
    uvicorn.run("app:app", host="0.0.0.0", port=8000, reload=False)

# 3. Start the server in a separate thread
thread = threading.Thread(target=run_server)
thread.start()

# 4. Start Cloudflared Tunnel
# This will give you a public URL (e.g., https://random-name.trycloudflare.com)
print("Starting Cloudflared Tunnel...")
!./cloudflared-linux-amd64 tunnel --url http://127.0.0.1:8000